# Step 09 — Model improvements

Tests, on the 2024 validation year: a stricter quality rule (chlorophyll below 0.01 mg/m³ treated as missing), seven new inputs (neighbourhood chlorophyll, recent change and anomaly, temperature anomaly and change, distance to the coast), predicting the correction to the blend rule, and light tuning. The winner, chosen by a rule fixed in advance (highest average gain over the blend on 2024), is then checked once on 2025 against the current model.

Nothing in steps 04–07 changes: the new model is saved as a candidate.

In [ ]:
# Step 09 — Model improvements. Tests new inputs, a stricter quality rule,
# predicting the correction to the blend rule, and light tuning, on 2024; then
# one final check of the winner on 2025. The production files (step 04's
# prepared data, step 05's model) are not changed.
from time import perf_counter

import joblib
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingRegressor

from chlforecast import baselines, config, samples, scoring
from chlforecast import features as F

TRAIN_SAMPLE_FRACTION = 0.3        # screening; the final fit uses FINAL_SAMPLE_FRACTION
FINAL_SAMPLE_FRACTION = 0.6        # as in step 05
HISTORY_START = "2016-12-01"       # a month before training starts, for 7-day histories
horizons = list(config.FORECAST_HORIZONS)

MODEL_PARAMS = dict(loss="squared_error", learning_rate=0.05, max_iter=150, max_leaf_nodes=31,
                    min_samples_leaf=100, l2_regularization=1.0, early_stopping=False, random_state=42)

current = joblib.load(config.STEP05_MODEL_PATH)   # the step 05 model, for the final comparison
print("Current model inputs:", current["feature_names"])

Current model inputs: ['season_sin', 'season_cos', 'chl_latest', 'chl_age_days', 'chl_mean_7d', 'chl_observations_7d', 'latitude_deg', 'longitude_deg', 'chl_clim_log', 'sst_c', 'sst_observed']


In [2]:
observations = xr.open_dataset(config.PROCESSED / "malta_daily_observations.nc").sel(time=slice(HISTORY_START, None))
dates = pd.DatetimeIndex(observations.time.values)
latitude = observations.latitude.values
longitude = observations.longitude.values
GRID = ("time", "latitude", "longitude")

raw_chl = observations["chl"].transpose(*GRID).values
sst = observations["sst_c"].transpose(*GRID).values

# Quality control: the existing upper bound, plus the lower bound tested here.
chl_old_rule = F.quality_control(raw_chl)
chl = F.quality_control(raw_chl, minimum=config.MIN_PLAUSIBLE_CHL)
del raw_chl

# Sea pixels: those the gap-filled product ever fills (as on the website maps).
with xr.open_dataset(config.DATA / "chl_l3_l4_comparison/chl_L4_2022-01-01_2025-12-31_13.9_15.2_35.55_36.9.nc") as l4:
    sea = np.isfinite(l4["CHL"].sortby("latitude").sortby("longitude").values).any(axis=0)

mask, _ = F.model_mask(chl, dates)
display(pd.Series({
    "Days": f"{dates[0]:%Y-%m-%d} to {dates[-1]:%Y-%m-%d}",
    "Values removed by the new lower bound": f"{int(np.isfinite(chl_old_rule).sum() - np.isfinite(chl).sum()):,}",
    "Modelled pixels": f"{int(mask.sum()):,} (step 04: {int((current['model_mask'].values == 1).sum()):,})",
}).to_frame("Value"))
del chl_old_rule

,Value
Days,2016-12-01 to 2025-12-31
Values removed by the new lower bound,177
Modelled pixels,"12,190 (step 04: 12,190)"


In [3]:
started = perf_counter()
monthly_log_clim = F.monthly_log_climatology(chl, dates, mask)
monthly_sst_clim = F.monthly_climatology(sst, dates, mask)

base = F.forecast_inputs(chl, sst, dates, mask, monthly_log_clim, latitude, longitude)
extra = F.extra_inputs(chl, sst, dates, base, sea, latitude, longitude, mask, monthly_sst_clim)
inputs = {**base, **extra}

targets = {}
for h in horizons:
    target = np.full(chl.shape, np.nan, dtype="float32")
    target[:-h] = chl[h:]
    targets[h] = target

# Issue days per period, stopping 7 days before each period ends (as in step 04).
last_target = dates + pd.Timedelta(days=max(horizons))
periods = {name: (dates >= pd.Timestamp(a)) & (last_target <= pd.Timestamp(b))
           for name, (a, b) in config.SPLIT_PERIODS.items()}
periods["train+validation"] = (dates >= pd.Timestamp(config.SPLIT_PERIODS["train"][0])) & (
    last_target <= pd.Timestamp(config.SPLIT_PERIODS["validation"][1]))

# Target-day climatology, for the blend rule.
target_clim = {h: F.climatology_for_dates(monthly_log_clim, dates + pd.Timedelta(days=h)) for h in horizons}

print(f"Built {len(inputs)} inputs for {len(dates):,} days in {(perf_counter() - started) / 60:.1f} min")
display(pd.DataFrame({
    name: {"Mean": float(np.nanmean(v[periods['train']])), "Missing (%)": float(100 * np.isnan(np.broadcast_to(v, chl.shape)[periods['train']][:, mask]).mean())}
    for name, v in extra.items()
}).T.round(3))

Built 18 inputs for 3,318 days in 0.1 min


,Mean,Missing (%)
chl_nbr5_yesterday,0.080,33.755
chl_nbr15_7d,0.084,0.470
chl_trend_log,-0.009,2.161
chl_anomaly_log,0.011,0.001
sst_anomaly,0.016,41.073
sst_change_7d,0.034,41.796
coast_distance_km,25.365,0.000


In [4]:
# Candidates. "Residual" models learn log10(observed) − log10(blend forecast):
# they start from the blend rule and only learn what it misses.
FEATURE_SETS = {
    "current": config.FINAL_FEATURES,
    "current + new": config.FINAL_FEATURES + config.EXTRA_FEATURES,
}
CANDIDATES = {
    "A · current inputs": ("current", False),
    "B · current inputs, residual": ("current", True),
    "C · current + new inputs": ("current + new", False),
    "D · current + new inputs, residual": ("current + new", True),
}


def rows(period, h, fraction, seed=0):
    r = samples.rows_from_arrays(inputs, targets[h], periods[period], mask, fraction, seed)
    r["persistence"] = np.asarray(inputs["chl_latest"])[r["t"], r["i"], r["j"]].astype("float64")
    r["clim_log"] = target_clim[h][r["t"], r["i"], r["j"]].astype("float64")
    return r


def blend_of(r, coefficients):
    return baselines.damped_persistence(coefficients, r["persistence"], r["clim_log"])


def fit_candidate(train, feature_set, residual, coefficients, params=MODEL_PARAMS):
    names = FEATURE_SETS[feature_set]
    y = np.log10(train["observed"])
    if residual:
        y = y - np.log10(blend_of(train, coefficients))
    model = HistGradientBoostingRegressor(**params).fit(train["X"][names], y)
    return {"model": model, "features": names, "residual": residual, "coefficients": coefficients}


def predict_candidate(fitted, r):
    out = fitted["model"].predict(r["X"][fitted["features"]])
    if fitted["residual"]:
        out = out + np.log10(blend_of(r, fitted["coefficients"]))
    return 10.0 ** out

In [5]:
# Screening on 2024: every candidate against the blend, with 95% ranges from
# resampling whole days.
screen_rows, screen_scores = [], []
for h in horizons:
    started = perf_counter()
    train = rows("train", h, TRAIN_SAMPLE_FRACTION)
    valid = rows("validation", h, 1.0)
    coefficients = baselines.fit_damped_persistence(train["observed"], train["persistence"], train["clim_log"])
    blend = blend_of(valid, coefficients)
    screen_scores.append({"Horizon": f"+{h}d", "Method": "Blend rule", **scoring.chl_scores(valid["observed"], blend)})
    for name, (feature_set, residual) in CANDIDATES.items():
        fitted = fit_candidate(train, feature_set, residual, coefficients)
        predicted = predict_candidate(fitted, valid)
        gain, low, high = scoring.log_rmse_gain_interval(valid["observed"], predicted, blend, valid["t"])
        screen_rows.append({"Horizon": f"+{h}d", "Candidate": name, "Gain over blend (%)": gain,
                            "95% low": low, "95% high": high})
        screen_scores.append({"Horizon": f"+{h}d", "Method": name, **scoring.chl_scores(valid["observed"], predicted)})
    print(f"+{h}d: {len(train['observed']):,} training rows, {(perf_counter() - started) / 60:.1f} min")
    del train, valid

screening = pd.DataFrame(screen_rows).set_index(["Horizon", "Candidate"])
display(screening.style.format("{:+.1f}"))
display(pd.DataFrame(screen_scores).set_index(["Horizon", "Method"])[["RMSE (log10)", "Bias (mg/m³)", "Median relative error (%)"]]
        .style.format({"RMSE (log10)": "{:.4f}", "Bias (mg/m³)": "{:+.4f}", "Median relative error (%)": "{:.1f}"}))

+1d: 5,425,586 training rows, 1.0 min
+4d: 5,423,327 training rows, 1.1 min
+7d: 5,425,316 training rows, 1.0 min


In [6]:
# Light tuning of the best candidate, on 2024. The rule, fixed before looking at
# 2025: the candidate with the highest average gain over the blend across the
# three lead times. To override, set BEST to a candidate name.
BEST = screening.groupby("Candidate")["Gain over blend (%)"].mean().idxmax()
print(f"Best candidate on 2024: {BEST}")
GRID_SEARCH = [
    dict(learning_rate=0.05, max_iter=150, max_leaf_nodes=31, min_samples_leaf=100),   # step 05 settings
    dict(learning_rate=0.05, max_iter=300, max_leaf_nodes=31, min_samples_leaf=100),
    dict(learning_rate=0.05, max_iter=300, max_leaf_nodes=63, min_samples_leaf=200),
    dict(learning_rate=0.10, max_iter=300, max_leaf_nodes=63, min_samples_leaf=500),
]
tune_rows = []
feature_set, residual = CANDIDATES[BEST]
for h in horizons:
    train = rows("train", h, TRAIN_SAMPLE_FRACTION)
    valid = rows("validation", h, 1.0)
    coefficients = baselines.fit_damped_persistence(train["observed"], train["persistence"], train["clim_log"])
    blend = blend_of(valid, coefficients)
    for k, settings in enumerate(GRID_SEARCH):
        fitted = fit_candidate(train, feature_set, residual, coefficients, {**MODEL_PARAMS, **settings})
        gain, low, high = scoring.log_rmse_gain_interval(valid["observed"], predict_candidate(fitted, valid), blend, valid["t"])
        tune_rows.append({"Horizon": f"+{h}d", "Settings": k, **settings, "Gain over blend (%)": gain, "95% low": low, "95% high": high})
    del train, valid
tuning = pd.DataFrame(tune_rows).set_index(["Horizon", "Settings"])
display(tuning.style.format({"Gain over blend (%)": "{:+.1f}", "95% low": "{:+.1f}", "95% high": "{:+.1f}"}))

Best candidate on 2024: C · current + new inputs


In [7]:
# The final check, once: refit the chosen candidate on 2017–2024 and score it on
# 2025 against the current model and the blend. Settings: those with the highest
# average gain on 2024 (same rule as above; override by setting FINAL_SETTINGS).
best_settings = int(tuning.groupby("Settings")["Gain over blend (%)"].mean().idxmax())
FINAL_SETTINGS = GRID_SEARCH[best_settings]
print(f"Final candidate: {BEST}, settings {best_settings}: {FINAL_SETTINGS}")
final, final_rows, final_scores, final_rows_by_area = {}, [], [], []
coast_km = F.coast_distance_km(sea, latitude, longitude)
for h in horizons:
    started = perf_counter()
    fit_rows = rows("train+validation", h, FINAL_SAMPLE_FRACTION)
    coefficients = baselines.fit_damped_persistence(fit_rows["observed"], fit_rows["persistence"], fit_rows["clim_log"])
    final[h] = fit_candidate(fit_rows, feature_set, residual, coefficients, {**MODEL_PARAMS, **FINAL_SETTINGS})
    del fit_rows

    test = rows("test", h, 1.0)
    predictions = {
        "New model": predict_candidate(final[h], test),
        "Current model (step 05)": 10.0 ** current["models"][h].predict(test["X"][current["feature_names"]]),
        "Blend rule": blend_of(test, coefficients),
    }
    for method, predicted in predictions.items():
        final_scores.append({"Horizon": f"+{h}d", "Method": method, **scoring.chl_scores(test["observed"], predicted)})
    for name, other in [("Blend rule", predictions["Blend rule"]), ("Current model (step 05)", predictions["Current model (step 05)"])]:
        gain, low, high = scoring.log_rmse_gain_interval(test["observed"], predictions["New model"], other, test["t"])
        final_rows.append({"Horizon": f"+{h}d", "New model vs": name, "Gain (%)": gain, "95% low": low, "95% high": high})
    near_coast = coast_km[test["i"], test["j"]] <= 5
    for area, sel in [("Within 5 km of the coast", near_coast), ("Further offshore", ~near_coast)]:
        row = {"Horizon": f"+{h}d", "Area": area, "Pixel-days": int(sel.sum())}
        for method, predicted in predictions.items():
            row[method] = scoring.chl_scores(test["observed"][sel], predicted[sel])["RMSE (log10)"]
        final_rows_by_area.append(row)
    print(f"+{h}d: fitted and tested in {(perf_counter() - started) / 60:.1f} min")
    del test

display(pd.DataFrame(final_rows).set_index(["Horizon", "New model vs"]).style.format("{:+.1f}"))
display(pd.DataFrame(final_scores).set_index(["Horizon", "Method"])[["RMSE (log10)", "Bias (mg/m³)", "Median relative error (%)"]]
        .style.format({"RMSE (log10)": "{:.4f}", "Bias (mg/m³)": "{:+.4f}", "Median relative error (%)": "{:.1f}"}))
display(pd.DataFrame(final_rows_by_area).set_index(["Horizon", "Area"]).style.format(
    {"Pixel-days": "{:,.0f}", "New model": "{:.4f}", "Current model (step 05)": "{:.4f}", "Blend rule": "{:.4f}"}))

Final candidate: C · current + new inputs, settings 0: {'learning_rate': 0.05, 'max_iter': 150, 'max_leaf_nodes': 31, 'min_samples_leaf': 100}
+1d: fitted and tested in 0.7 min
+4d: fitted and tested in 0.7 min
+7d: fitted and tested in 0.6 min


In [8]:
# Save the new model as a candidate. The model in use is not replaced.
from datetime import datetime, timezone
candidate_path = config.BASE / "models" / "chl_forecast_candidate_step09.joblib"
joblib.dump({
    "models": {h: final[h]["model"] for h in horizons},
    "feature_names": list(final[horizons[0]]["features"]),
    "residual_on_blend": residual,
    "damped_persistence_coefficients": {h: list(final[h]["coefficients"]) for h in horizons},
    "model_settings": {**MODEL_PARAMS, **FINAL_SETTINGS},
    "min_plausible_chl": config.MIN_PLAUSIBLE_CHL,
    "monthly_log_climatology": monthly_log_clim,
    "monthly_sst_climatology": monthly_sst_clim,
    "model_mask": mask,
    "screening": screening.reset_index().to_dict("records"),
    "test_gains": final_rows,
    "test_scores": final_scores,
    "created_utc": datetime.now(timezone.utc).isoformat(),
}, candidate_path, compress=3)
print(f"Saved candidate: {candidate_path}")

Saved candidate: /Users/alangatt/git/mcast.pages.chl_prediction/experiment_v2/models/chl_forecast_candidate_step09.joblib
